# Projected conv · ARC-AGI-2 · attached source folder

Enable GPU and keep Internet OFF. Attach the official competition data and your `symarc-cide` Dataset. This notebook uses `/kaggle/input/datasets/teddlo/symarc-cide/src`, copies code to writable storage, and includes its runner and small offline dependency wheels. No source ZIP or Git clone is required.

Run the smoke test before full training. MAX_TASKS=None is configured to attempt all tasks. Up to four visible GPUs process independent tasks. Check smoke_test_result.json and submission_validation.json for executed evidence; notebook source alone does not establish successful execution. Full coverage within Kaggle's time limit is not guaranteed; unprocessed tasks receive explicitly audited fallback guesses. Final output is `/kaggle/working/submission.json`.


In [ ]:
from pathlib import Path
import base64, hashlib, importlib.util, json, os, shutil, subprocess, sys, time
import torch

SOURCE_DIR = Path('/kaggle/input/datasets/teddlo/symarc-cide/src')
CHALLENGES_PATH = None  # Set to the exact official ARC-AGI-2 test challenges path if ambiguous.
STEPS = 1500
SEED = 0
MAX_TASKS = None  # Attempt every task; optional 1 for an initial timing experiment.
TOTAL_BUDGET_HOURS = 6
PER_TASK_LIMIT_HOURS = 2
KEEP_CHECKPOINTS = False
WORKFLOW_START = time.monotonic()

assert torch.cuda.is_available(), 'Enable a GPU accelerator in Kaggle settings.'
assert (SOURCE_DIR/'train.py').is_file(), f'Cannot find {SOURCE_DIR}/train.py'
assert (SOURCE_DIR/'config/models/final_projected_conv_20.yaml').is_file(), 'Upload the current ARC-fafo src folder.'
GPU_WORKERS=min(4,torch.cuda.device_count())
print('GPU workers:',GPU_WORKERS,'PyTorch:',torch.__version__,'CUDA:',torch.version.cuda)
print('Four-way parallelism requires four visible GPUs; this runtime exposes',torch.cuda.device_count())
if CHALLENGES_PATH is not None:
    challenges_path=Path(CHALLENGES_PATH)
else:
    # Exclude ARC-AGI-1 data bundled with the attached code dataset.
    matches=[f for f in Path('/kaggle/input').rglob('arc-agi_test_challenges.json')
             if not f.is_relative_to(SOURCE_DIR.parent)]
    if len(matches)!=1:
        raise RuntimeError(f'Set CHALLENGES_PATH to the official ARC-AGI-2 test file. Candidates: {matches}')
    challenges_path=matches[0]
assert challenges_path.is_file(), challenges_path
assert not challenges_path.is_relative_to(SOURCE_DIR.parent), 'Use the official competition dataset, not the code dataset.'
WORK=Path('/kaggle/working/projected_conv_work')
if WORK.exists():
    raise RuntimeError('WORK already exists. Start a fresh session or choose a fresh WORK directory.')
WORK.mkdir()
SRC=WORK/'src'
shutil.copytree(SOURCE_DIR,SRC,ignore=shutil.ignore_patterns('dataset','tests','notebook','__pycache__','.pytest_cache','wandb'))
(WORK/'run_arcagi2.py').write_text('"""Per-task projected-conv training and two-attempt ARC submission export."""\nimport argparse\nimport copy\nimport json\nimport os\nimport subprocess\nimport sys\nimport time\nimport hashlib\nimport threading\nfrom concurrent.futures import ThreadPoolExecutor\nfrom queue import Queue, Empty\nfrom pathlib import Path\n\n\ndef atomic_json(path, data):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(data), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef valid_grid(grid):\n    return (isinstance(grid, (list, tuple)) and 1 <= len(grid) <= 30\n            and all(isinstance(row, (list, tuple)) and len(row) == len(grid[0]) for row in grid)\n            and 1 <= len(grid[0]) <= 30\n            and all(type(v) is int and 0 <= v <= 9 for row in grid for v in row))\n\n\ndef main():\n    p = argparse.ArgumentParser(description=__doc__)\n    p.add_argument(\'--source\', type=Path, required=True, help=\'Writable source copy containing train.py\')\n    p.add_argument(\'--challenges\', type=Path, required=True)\n    p.add_argument(\'--output\', type=Path, required=True)\n    p.add_argument(\'--steps\', type=int, default=1500)\n    p.add_argument(\'--seed\', type=int, default=0)\n    p.add_argument(\'--max-tasks\', type=int)\n    p.add_argument(\'--task\', action=\'append\')\n    p.add_argument(\'--budget-seconds\', type=float, default=6*3600)\n    p.add_argument(\'--per-task-seconds\', type=float, default=2*3600)\n    p.add_argument(\'--device\', default=\'cuda:0\')\n    p.add_argument(\'--gpus\', type=int, default=1, help=\'Independent task workers; one process per GPU\')\n    p.add_argument(\'--keep-checkpoints\', action=\'store_true\')\n    p.add_argument(\'--wandb-mode\', choices=[\'disabled\',\'offline\'], default=\'disabled\')\n    p.add_argument(\'--wandb-tag\', default=\'kaggle run 1\')\n    a = p.parse_args()\n    if a.steps < 1 or a.budget_seconds <= 0 or a.per_task_seconds <= 0:\n        p.error(\'steps and time limits must be positive\')\n    if a.max_tasks is not None and a.max_tasks < 1:\n        p.error(\'max-tasks must be positive\')\n    if not 1 <= a.gpus <= 4:\n        p.error(\'gpus must be 1..4\')\n    if a.device.startswith(\'cuda\'):\n        import torch\n        first_device = int(a.device.split(\':\')[1]) if \':\' in a.device else 0\n        if not torch.cuda.is_available() or first_device+a.gpus > torch.cuda.device_count():\n            p.error(\'Requested GPUs are not visible in this runtime\')\n        devices = [f\'cuda:{first_device+i}\' for i in range(a.gpus)]\n    else:\n        devices = [a.device]*a.gpus\n    source = a.source.resolve()\n    out = a.output.resolve()\n    out.mkdir(parents=True, exist_ok=True)\n    if (out/\'submission.json\').exists():\n        raise SystemExit(\'Output already contains submission.json; choose a fresh output directory.\')\n    problems = json.loads(a.challenges.read_text())\n    if not isinstance(problems, dict) or not problems:\n        raise ValueError(\'Expected task-ID -> challenge mapping\')\n    # Even if public task JSON contains answers, never pass test labels to the model.\n    problems = copy.deepcopy(problems)\n    for task, problem in problems.items():\n        if not problem.get(\'train\') or not problem.get(\'test\'):\n            raise ValueError(f\'Invalid challenge: {task}\')\n        for example in problem[\'test\']:\n            example.pop(\'output\', None)\n    dataset = source/\'dataset\'\n    dataset.mkdir(exist_ok=True)\n    atomic_json(dataset/\'arc-agi_test_challenges.json\', problems)\n    selected = a.task or list(problems)\n    if len(set(selected)) != len(selected) or not set(selected) <= set(problems):\n        raise ValueError(\'Selected task IDs must be distinct and present in challenges\')\n    if a.max_tasks is not None:\n        selected = selected[:a.max_tasks]\n    # A valid, explicitly reported fallback for tasks not completed within budget.\n    submission = {t:[{\'attempt_1\':copy.deepcopy(e[\'input\']), \'attempt_2\':copy.deepcopy(e[\'input\'])}\n                     for e in problem[\'test\']] for t,problem in problems.items()}\n    audit = {t:dict(status=\'fallback_not_attempted\', predictions=\'copy_input_fallback\') for t in problems}\n    atomic_json(out/\'submission.json\', submission)\n    atomic_json(out/\'task_status.json\', audit)\n    env = os.environ.copy()\n    env.update(OMP_NUM_THREADS=\'4\', MKL_NUM_THREADS=\'4\', PYTHONUNBUFFERED=\'1\', WANDB_MODE=a.wandb_mode)\n    if a.wandb_mode == \'offline\':\n        (out/\'wandb\').mkdir(exist_ok=True)\n        env[\'WANDB_DIR\'] = str(out/\'wandb\')\n        env[\'WANDB_DATA_DIR\'] = str(out/\'wandb_data\')\n        env[\'WANDB_CACHE_DIR\'] = str(out/\'wandb_cache\')\n    manifest = dict(source=str(source),challenges=str(a.challenges),steps=a.steps,seed=a.seed,\n                    device=a.device,selected_tasks=selected,budget_seconds=a.budget_seconds,\n                    per_task_seconds=a.per_task_seconds,devices=devices,source_metadata=None)\n    meta = source.parent/\'source_manifest.json\'\n    if meta.exists():\n        manifest[\'source_metadata\'] = json.loads(meta.read_text())\n    atomic_json(out/\'run_manifest.json\', manifest)\n    start = time.monotonic()\n    state_lock = threading.Lock()\n    def run_one(task, device):\n        remaining = a.budget_seconds-(time.monotonic()-start)-30\n        if remaining <= 0:\n            return False\n        checkpoint_dir = out/\'checkpoints\'/task\n        checkpoint = checkpoint_dir/f\'{task}.pt\'\n        cmd = [sys.executable,str(source/\'train.py\'),\'--config\',str(source/\'config/models/final_projected_conv_20.yaml\'),\n               \'--task\',task,\'--set\',\'training.split=test\',\'--set\',f\'training.iterations={a.steps}\',\n               \'--set\',f\'training.seed={a.seed}\',\'--set\',f\'training.device={device}\',\n               \'--set\',\'logging.wandb=\'+str(a.wandb_mode != \'disabled\').lower(),\n               \'--set\',f\'logging.wandb_mode={a.wandb_mode}\',\n               \'--set\',\'logging.wandb_tags=\'+json.dumps([a.wandb_tag]),\'--save-checkpoints\',\'--output-dir\',str(checkpoint_dir)]\n        print(f\'Training {task} on {device}: {a.steps} steps, timeout {min(remaining,a.per_task_seconds):.0f}s\',flush=True)\n        task_start = time.monotonic()\n        log_path = out/f\'{task}.log\'\n        try:\n            with log_path.open(\'w\') as log:\n                subprocess.run(cmd,cwd=source,env=env,stdout=log,stderr=subprocess.STDOUT,\n                               check=True,timeout=min(remaining,a.per_task_seconds))\n            import torch\n            payload = torch.load(checkpoint,map_location=\'cpu\',weights_only=True)\n            first = payload[\'selection\'][\'top_1\']\n            second = payload[\'selection\'][\'top_2\']\n            if first is None or len(first) != len(problems[task][\'test\']):\n                raise ValueError(\'Incorrect number of selected test grids\')\n            if second is None:\n                second = first\n            if len(second) != len(first):\n                raise ValueError(\'Incorrect number of second attempts\')\n            guesses = []\n            for grid1, grid2 in zip(first,second):\n                if not valid_grid(grid1) or not valid_grid(grid2):\n                    raise ValueError(\'Invalid selected grid\')\n                guesses.append({\'attempt_1\':[list(row) for row in grid1], \'attempt_2\':[list(row) for row in grid2]})\n            result = dict(status=\'model_completed\',predictions=\'projected_conv\',completed_steps=payload[\'completed_steps\'])\n            if not a.keep_checkpoints:\n                checkpoint.unlink()\n        except subprocess.TimeoutExpired:\n            result = dict(status=\'fallback_timeout\',predictions=\'copy_input_fallback\')\n        except Exception as exc:\n            result = dict(status=\'fallback_error\',predictions=\'copy_input_fallback\',error=str(exc),log=str(log_path))\n        result.update(elapsed_seconds=time.monotonic()-task_start,device=device)\n        with state_lock:\n            if result[\'status\'] == \'model_completed\':\n                submission[task] = guesses\n            audit[task] = result\n            atomic_json(out/\'submission.json\', submission)\n            atomic_json(out/\'task_status.json\', audit)\n        print(task, result,flush=True)\n        return True\n    pending = Queue()\n    for task in selected:\n        pending.put(task)\n    def worker(device):\n        while True:\n            try:\n                task = pending.get_nowait()\n            except Empty:\n                return\n            if not run_one(task,device):\n                return\n    with ThreadPoolExecutor(max_workers=len(devices)) as executor:\n        futures = [executor.submit(worker,device) for device in devices]\n        for future in futures:\n            future.result()\n    assert set(submission) == set(problems)\n    for task, attempts in submission.items():\n        assert len(attempts) == len(problems[task][\'test\'])\n        for pair in attempts:\n            assert set(pair) == {\'attempt_1\',\'attempt_2\'}\n            assert valid_grid(pair[\'attempt_1\']) and valid_grid(pair[\'attempt_2\'])\n    n = sum(r[\'status\']==\'model_completed\' for r in audit.values())\n    atomic_json(out/\'submission_validation.json\',dict(schema_valid=True,tasks=len(problems),\n                test_inputs=sum(len(p[\'test\']) for p in problems.values()),model_completed=n,\n                fallback_tasks=len(problems)-n,full_model_coverage=n==len(problems),\n                submission_sha256=hashlib.sha256((out/\'submission.json\').read_bytes()).hexdigest()))\n    print(f\'Model completed {n}/{len(problems)} tasks. Remaining tasks have explicitly logged fallback outputs.\',flush=True)\n    print(\'Submission:\',out/\'submission.json\',flush=True)\n\n\nif __name__ == \'__main__\':\n    main()\n',encoding='utf-8')
wheel_data={'antlr4_python3_runtime-4.9.3-py3-none-any.whl': '...', 'omegaconf-2.3.0-py3-none-any.whl': '...'}
(WORK/'wheels').mkdir()
for name,encoded in wheel_data.items():
    (WORK/'wheels'/name).write_bytes(base64.b64decode(encoded))
source_manifest=dict(source=str(SOURCE_DIR),files={f.relative_to(SRC).as_posix():hashlib.sha256(f.read_bytes()).hexdigest()
                      for f in SRC.rglob('*') if f.is_file()})
(WORK/'source_manifest.json').write_text(json.dumps(source_manifest,indent=2))
print('Code copied into:',SRC)
print('Official challenges:',challenges_path)
print('Runner and offline wheels extracted from this notebook.')


In [ ]:
# Keep Kaggle's GPU-enabled PyTorch. Install only small missing dependencies offline.
import yaml, numpy, tqdm, matplotlib
assert importlib.util.find_spec('wandb') is not None, 'Your Kaggle image needs wandb installed; attach offline wheels if absent.'
if importlib.util.find_spec('omegaconf') is None or importlib.util.find_spec('antlr4') is None:
    subprocess.run([sys.executable, '-m', 'pip', 'install', '--no-index', '--find-links',
                    str(WORK/'wheels'), 'omegaconf==2.3.0', 'antlr4-python3-runtime==4.9.3'], check=True)
from omegaconf import OmegaConf
print('OmegaConf available. Existing PyTorch retained:', torch.__version__)
problems = json.loads(challenges_path.read_text())
assert isinstance(problems, dict) and problems
assert all(p.get('train') and p.get('test') for p in problems.values())
print('Tasks:', len(problems), 'Test inputs:', sum(len(p['test']) for p in problems.values()))
Path('/kaggle/working/runtime_versions.json').write_text(json.dumps(dict(
    python=sys.version, torch=torch.__version__, cuda=torch.version.cuda,
    numpy=numpy.__version__, gpu=torch.cuda.get_device_name(0),
    challenges_sha256=hashlib.sha256(challenges_path.read_bytes()).hexdigest()), indent=2))


## 1. Smoke test
Runs two optimizer steps on one task per GPU worker, exercising GPU preprocessing, projected-conv forward/backward, candidate selection, checkpoint save/read, and submission export. It writes only to `smoke/`; it is not the final submission. If this fails, inspect its task log before proceeding.


In [ ]:
SMOKE = WORK/'smoke'
smoke_start = time.monotonic()
subprocess.run([sys.executable, str(WORK/'run_arcagi2.py'), '--source', str(SRC),
                '--challenges', str(challenges_path), '--output', str(SMOKE),
                '--steps', '2', '--seed', str(SEED), '--max-tasks', str(GPU_WORKERS),
                '--budget-seconds', '600', '--per-task-seconds', '540', '--gpus', str(GPU_WORKERS)], check=True)
smoke_status = json.loads((SMOKE/'task_status.json').read_text())
smoke_completed = [t for t,s in smoke_status.items() if s['status']=='model_completed']
if len(smoke_completed) != min(GPU_WORKERS, len(problems)):
    for log in SMOKE.glob('*.log'):
        print(log.read_text()[-12000:])
    raise RuntimeError('Smoke test failed; do not run the full workload yet.')
smoke_result = dict(status='passed', tasks=smoke_completed, gpu_workers=GPU_WORKERS, devices=sorted({smoke_status[t]['device'] for t in smoke_completed}), elapsed_seconds=time.monotonic()-smoke_start)
Path('/kaggle/working/smoke_test_result.json').write_text(json.dumps(smoke_result, indent=2))
print('Smoke test passed:', smoke_result)


## 2. Train and generate guesses
A shared task queue dispatches independent tasks to up to four GPUs. The task order follows the challenges JSON, without performance filtering. Check the actual competition's hardware and runtime limits before choosing the budget. The default 1,500-step configuration is unchanged, but PyTorch/CUDA/hardware can differ from M3 and affect the trajectory. Training starts from scratch for every task; the script does not resume training from inference checkpoints.


In [ ]:
RUN = WORK/'predictions'
remaining = TOTAL_BUDGET_HOURS*3600 - (time.monotonic()-WORKFLOW_START) - 120
assert remaining > 60, 'Insufficient remaining budget.'
command = [sys.executable, str(WORK/'run_arcagi2.py'), '--source', str(SRC),
           '--challenges', str(challenges_path), '--output', str(RUN),
           '--steps', str(STEPS), '--seed', str(SEED),
           '--budget-seconds', str(remaining), '--per-task-seconds', str(PER_TASK_LIMIT_HOURS*3600), '--gpus', str(GPU_WORKERS)]
command += ['--wandb-mode', 'offline', '--wandb-tag', 'kaggle run 1', '--keep-checkpoints']
if MAX_TASKS is not None:
    command += ['--max-tasks', str(MAX_TASKS)]
if KEEP_CHECKPOINTS:
    command += ['--keep-checkpoints']
print('Starting:', command)
subprocess.run(command, check=True)


## 3. Validate and publish `submission.json`
There must be two attempts for every test input. Completed tasks use the tracker's final top two, not reference-answer-based selection. Unprocessed, timed-out, or failed tasks use copy-input fallback guesses and are counted explicitly. The fallback only keeps the file structurally complete; it does not indicate that the model processed the task.


In [ ]:
submission = json.loads((RUN/'submission.json').read_text())
status = json.loads((RUN/'task_status.json').read_text())
assert set(submission) == set(problems)
for task, examples in problems.items():
    assert len(submission[task]) == len(examples['test'])
    assert all(set(pair)=={'attempt_1','attempt_2'} for pair in submission[task])
sample_candidates = list(challenges_path.parent.glob('*sample_submission*.json'))
if len(sample_candidates) == 1:
    sample = json.loads(sample_candidates[0].read_text())
    assert set(sample) == set(submission), 'Sample submission and challenges task IDs differ.'
    assert all(len(sample[t])==len(submission[t]) for t in sample)
completed = sum(s['status']=='model_completed' for s in status.values())
print(f'MODEL COVERAGE: {completed}/{len(problems)} tasks; FALLBACK: {len(problems)-completed}.')
assert completed > 0, 'No task completed; refusing to publish an all-fallback submission.'
shutil.copy2(RUN/'submission.json', '/kaggle/working/submission.json')
shutil.copy2(RUN/'task_status.json', '/kaggle/working/task_status.json')
shutil.copy2(RUN/'run_manifest.json', '/kaggle/working/run_manifest.json')
shutil.copy2(RUN/'submission_validation.json', '/kaggle/working/submission_validation.json')
print('Validation:', json.loads((RUN/'submission_validation.json').read_text()))
print('Offline logs:', shutil.make_archive('/kaggle/working/kaggle_run_1_logs', 'zip', RUN))
print('Output ready: /kaggle/working/submission.json')
print('Save a notebook version, inspect task_status.json and logs, then use Kaggle submission controls.')
